# E7 step 1 · The chain's first counting number, on the validation images (on this PC, CPU)

Task **E7**, step 1, from `Essential_task_delegation/Next_Steps_Roadmap.md`. Built 7 Nov 2026. It uses saved files only: no GPU, no training, nothing from the test pairs.

**The chain being scored:** SAM 3 (a text-prompted segmentation model, prompt "building") outlines each building on the before-photo, cut into four quarters. A Siamese ResNet-18 (128 px before and after crops of one building, trained on SAM 3's outlines of the training images) labels each outline intact, damaged, destroyed or not a building. Outlines labelled "not a building" are dropped. This is R16c step 5's SAM 3-only map, the one that reached the destroyed-building target on clean validation data.

**Why**
- No counting figure exists for this map. R16c step 1 counted older maps, and the plan now needs this chain's number.
- It sets the expectation for step 2 (`e7-step2-chain-on-test-pairs.ipynb`) before any test score exists.

**How it counts** (R16c step 1's method, H2's table rule)
- Paint each outline with its label, using each grader's own label biases from `calibration.json`.
- Count the 8-connected blobs of each class, of at least 20 px. DisasterM3's true count is the number of 8-connected blobs, so the true masks must answer 100% of the questions.
- Pick the option nearest the count.
- Questions: the 1,055 training-split counting questions on the 645 validation images.

**Scores, and why**
- **Accuracy on the options:** the measure the review's target uses.
- **Count error against the true count:** added 7 Nov, because a rule that never sees the image scores 66.98% on the benchmark's optical questions (`benchmark-option-rule-check.ipynb`). Accuracy alone can't show that a method counts well.
- **The no-image rule on the same questions:** shows how much of the accuracy the options alone could give. These are training questions, and the rule was fixed on them, so here it's only a reference.

**Checks**
- Painting the saved labels must rebuild the map's published pixel scores (mIoU, destroyed-building IoU).
- Every outline must match its row in `instances_val.csv` on its true pixels.
- The true masks must answer 100% of the questions.

**Expectation, fixed before this run (7 Nov):** about 36–40% accuracy, near R11's 39.24% (R11: U-Net with a ResNet-50 encoder, before and after photos as input). The older SAM 3-only map with R16b's grades scored 36.40%, and a perfect grader on the same outlines 47.39%. Count error should be about R11's (median 33%).

**How to run:** in Jupyter on this PC, Run All (a few minutes). Results go to `E7-results/step1/`.


In [1]:
# ── Cell 1: Configuration and inputs (CPU only, no installs needed) ─────────
import ast
import json
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image                # PIL and SciPy, as in R16c step 1: this PC has no OpenCV
from scipy import ndimage
from scipy.stats import binomtest
from tqdm.auto import tqdm

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "train_release.json").exists())
CT = REPO / "Essential_task_delegation" / "Champion_Task"
MANIFEST_PATH = REPO / "train_release.json"
SAM3_DIR = CT / "E3-results" / "E3" / "masks" / "pre_grid2"   # E3's SAM 3 instance maps (16-bit, 0 = background)
GT_DIR = CT / "E1-results" / "E1" / "gt"                       # the 645 validation masks (0-3), saved by E1
STEP5, STEP5B = CT / "R16c-results" / "step5", CT / "R16c-results" / "step5b"
R11_FILE = CT / "R16c-results" / "step1" / "R16c_counting_per_question.csv"
LEAK_FILE = CT / "E5-results" / "E5" / "leaked_pairs_40.csv"
OUT_DIR = CT / "E7-results" / "step1"

GRADES = ["Intact", "Damaged", "Destroyed"]
LABELS = GRADES + ["None"]
MIN_AREA = 20                        # R16's smallest building piece, also H2's table rule
SEED_DIR = {42: STEP5, 43: STEP5, 44: STEP5B}
SUMMARY = {42: STEP5 / "R16c_step5_summary.csv", 43: STEP5 / "R16c_step5_summary.csv",
           44: STEP5B / "R16c_step5b_summary.csv"}
CLASS_OF = {"Intact building counting.": 1, "Damaged building counting.": 2,
            "Totally-destroyed building counting.": 3}
R11_NAME = "R11, blobs of at least 20 px"
R11_ACCURACY = 39.24                 # R16c step 1

OUT_DIR.mkdir(parents=True, exist_ok=True)
t_start = time.time()
BIASES = {}
for folder in (STEP5, STEP5B):
    for seed, chosen in json.loads((folder / "calibration.json").read_text(encoding="utf-8")).items():
        BIASES[int(seed)] = np.array([chosen["bias"][l] for l in LABELS])
LEAK40 = set(pd.read_csv(LEAK_FILE)["validation_pair"])
assert len(LEAK40) == 40 and sorted(BIASES) == [42, 43, 44]
val_files = sorted(p.name for p in GT_DIR.glob("*.png"))
assert len(val_files) == 645
print(f"✓ E7 step 1 on this PC: {REPO}")
print(f"  {len(val_files)} validation images, {len(LEAK40)} of them E5's leaked pairs; biases "
      f"{ {s: b.tolist() for s, b in BIASES.items()} }")


✓ E7 step 1 on this PC: D:\CSE499AB_project
  645 validation images, 40 of them E5's leaked pairs; biases {42: [0.0, -1.75, -4.0, -1.25], 43: [0.0, -1.0, -4.0, -2.0], 44: [0.0, -2.0, -4.0, -3.5]}


In [2]:
# ── Cell 2: Paint each grader's labels on SAM 3's outlines, check the map, count the blobs ──
EIGHT = np.ones((3, 3), bool)
CLASSES = (1, 2, 3)

def blobs(binary, min_area=0):
    """Number of 8-connected blobs (DisasterM3's counting rule); min_area > 0 drops smaller ones."""
    labels, n = ndimage.label(binary, structure=EIGHT)
    if min_area == 0 or n == 0:
        return n
    return int((np.bincount(labels.ravel())[1:] >= min_area).sum())

def confusion(gt, pred):
    """4×4 pixel counts, rows = truth, columns = prediction."""
    return np.bincount(gt.astype(np.int64).ravel() * 4 + pred.astype(np.int64).ravel(), minlength=16).reshape(4, 4)

def scores(cm):
    cm = cm.astype(np.float64)
    tp = np.diag(cm)
    iou = tp / (cm.sum(0) + cm.sum(1) - tp + 1e-8)
    return {"miou": round(float(iou.mean()), 4), "destroyed_iou": round(float(iou[3]), 4)}

# Self-checks: a diagonal touch is one blob; the area rule drops small blobs; a perfect map scores 1
_m = np.zeros((6, 6), bool)
_m[1, 1] = _m[2, 2] = True
_m[4, 3:6] = True
assert blobs(_m) == 2 and blobs(_m, min_area=3) == 1
_g = np.array([[0, 1], [2, 3]], np.uint8)
assert scores(confusion(_g, _g)) == {"miou": 1.0, "destroyed_iou": 1.0}

GT_COLS = ["gt_background_px", "gt_intact_px", "gt_damaged_px", "gt_destroyed_px"]
tables = {}
for seed in BIASES:
    inst = pd.read_csv(SEED_DIR[seed] / "instances_val.csv")
    inst = inst[inst["found_by"] == "sam3"]
    logp = inst[[f"logp_{l.lower()}_s{seed}" for l in LABELS]].to_numpy(np.float64)
    label = (logp + BIASES[seed]).argmax(1)
    inst = inst.assign(cls=np.where(label < 3, label + 1, 0))     # "None" stays background
    tables[seed] = {name: rows for name, rows in inst.groupby("image")}

cms = {seed: {"all": np.zeros((4, 4), np.int64), "clean": np.zeros((4, 4), np.int64)} for seed in BIASES}
counts, truth_counts, unmatched = {}, {}, 0
for name in tqdm(val_files, desc="Validation images"):
    gt = np.array(Image.open(GT_DIR / name).convert("L"))
    sam = np.array(Image.open(SAM3_DIR / name)).astype(np.int64)
    assert sam.shape == gt.shape, f"{name}: map sizes differ"
    size = int(sam.max()) + 1
    gt_px = np.bincount(sam.ravel() * 4 + gt.ravel(), minlength=size * 4).reshape(size, 4)
    present = np.unique(sam)
    present = present[present > 0]
    truth_counts[name] = [blobs(gt == c) for c in CLASSES]
    for seed in BIASES:
        rows = tables[seed].get(name)
        lut = np.zeros(size, np.uint8)
        seen = np.zeros(size, bool)
        if rows is not None:
            ids = rows["id"].to_numpy()
            lut[ids] = rows["cls"].to_numpy()
            seen[ids] = True
            unmatched += int((gt_px[ids] != rows[GT_COLS].to_numpy()).any(1).sum())
        unmatched += int((~seen[present]).sum())
        painted = lut[sam]
        cm = confusion(gt, painted)
        cms[seed]["all"] += cm
        if name not in LEAK40:
            cms[seed]["clean"] += cm
        counts[(name, seed)] = [blobs(painted == c, MIN_AREA) for c in CLASSES]
assert unmatched == 0, f"{unmatched} outlines don't match instances_val.csv"
print(f"✓ Every outline matches its row in instances_val.csv ({sum(len(r) for r in tables[42].values()):,} outlines)")

# The rebuilt maps must give the published scores. Seed 43's saved log-probabilities have 4 decimals, and step 5b
# measured a rebuild difference of up to 0.0012 for it, so the limit is 0.002.
check = []
for seed in BIASES:
    pub = pd.read_csv(SUMMARY[seed])
    pub = pub[(pub["variant"] == f"R16c_s{seed}_sam3") & (pub["source"] == "all")].iloc[0]
    mine, clean = scores(cms[seed]["all"]), scores(cms[seed]["clean"])
    check.append({"seed": seed, "miou": mine["miou"], "published_miou": pub["miou"], "destroyed_iou": mine["destroyed_iou"],
                  "published_destroyed_iou": pub["destroyed_iou"], "clean_miou": clean["miou"],
                  "clean_destroyed_iou": clean["destroyed_iou"]})
check = pd.DataFrame(check)
print(check.to_string(index=False))
worst = float(np.abs(check[["miou", "destroyed_iou"]].to_numpy() - check[["published_miou", "published_destroyed_iou"]].to_numpy()).max())
assert worst <= 0.002, f"The rebuilt map differs from the published score by {worst}"
print(f"✓ The rebuilt maps give the published scores (largest difference {worst:.4f})")


Validation images:   0%|          | 0/645 [00:00<?, ?it/s]

✓ Every outline matches its row in instances_val.csv (26,834 outlines)
 seed   miou  published_miou  destroyed_iou  published_destroyed_iou  clean_miou  clean_destroyed_iou
   42 0.5520          0.5520         0.3347                   0.3347      0.5469               0.3161
   43 0.5525          0.5522         0.3254                   0.3242      0.5473               0.3063
   44 0.5528          0.5527         0.3156                   0.3156      0.5480               0.2982
✓ The rebuilt maps give the published scores (largest difference 0.0012)


In [3]:
# ── Cell 3: The 1,055 questions, every method's answers, accuracy and count error ──
def name_of(path):
    return Path(str(path).replace("\\", "/")).name

def options_of(e):
    o = e["options_list"]
    return [float(x) for x in (ast.literal_eval(o) if isinstance(o, str) else o)]

def nearest_option(n, options):
    """H1 V1's rule: the option closest to the count (the first one on a tie)."""
    return "ABCDE"[min(range(len(options)), key=lambda i: abs(options[i] - n))]

def no_image_rule(options):
    """Never sees the image: largest option above 20 -> the middle value, otherwise the second smallest."""
    s = sorted(options)
    return "ABCDE"[options.index(s[2] if s[4] > 20 else s[1])]

assert nearest_option(52, [10, 40, 60, 80, 100]) == "C" and no_image_rule([231, 323, 277, 139, 185]) == "A"
with open(MANIFEST_PATH, encoding="utf-8") as f:
    raw = json.load(f)
val_set = set(val_files)
questions = [e for e in raw if e.get("task") == "Building Damage Counting" and name_of(e["post_image_path"]) in val_set]
del raw
assert len(questions) == 1055, f"{len(questions):,} counting questions; R16c step 1 had 1,055"
r11 = pd.read_csv(R11_FILE)
assert len(r11) == 1055

rows, truth_right = [], 0
for i, e in enumerate(questions):
    name, cls = name_of(e["post_image_path"]), CLASS_OF[e["cls_description"]]
    opts, right, truth = options_of(e), e["ground_truth_option"].strip()[0], float(e["ground_truth"])
    assert r11.loc[i, "image"] == name and float(r11.loc[i, "truth"]) == truth, "R16c step 1's rows are in another order"
    truth_right += nearest_option(truth_counts[name][cls - 1], opts) == right
    row = {"image": name, "class": GRADES[cls - 1].lower(), "truth": truth, "truth_option": right,
           "clean": name not in LEAK40, "largest_option": max(opts),
           "no_image_rule | correct": no_image_rule(opts) == right,
           "always_middle | correct": "ABCDE"[opts.index(sorted(opts)[2])] == right}
    for seed in BIASES:
        n = counts[(name, seed)][cls - 1]
        row[f"chain_s{seed} | count"] = n
        row[f"chain_s{seed} | correct"] = nearest_option(n, opts) == right
    row[f"R11 | count"] = int(r11.loc[i, f"{R11_NAME} | count"])
    row[f"R11 | correct"] = nearest_option(row["R11 | count"], opts) == right
    rows.append(row)
per_q = pd.DataFrame(rows)
assert truth_right == 1055, f"The true masks answer {truth_right} of 1,055 questions: the counting rule is off"
r11_acc = round(100 * per_q["R11 | correct"].mean(), 2)
assert r11_acc == R11_ACCURACY, f"R11 answers {r11_acc}%; R16c step 1 had {R11_ACCURACY}%"
print(f"✓ The true masks answer all 1,055 questions; R11 reproduces {r11_acc}%")

def count_error(df, m):
    err = (df[f"{m} | count"] - df["truth"]).abs()
    pct = err / df["truth"].clip(lower=1)
    return {"mean_abs_error": round(float(err.mean()), 2), "median_abs_error": float(err.median()),
            "median_error_%": round(100 * float(pct.median()), 1), "within_20%_%": round(100 * float((pct <= 0.2).mean()), 1)}

def accuracy(df, m):
    return round(100 * df[f"{m} | correct"].mean(), 2)

big = per_q["largest_option"] > 20
table = []
for m in [f"chain_s{s}" for s in BIASES] + ["R11", "no_image_rule", "always_middle"]:
    row = {"method": m, "all_%": accuracy(per_q, m), "clean_%": accuracy(per_q[per_q["clean"]], m),
           **{f"{c}_%": accuracy(per_q[per_q["class"] == c], m) for c in ("intact", "damaged", "destroyed")},
           "largest_above_20_%": accuracy(per_q[big], m), "largest_up_to_20_%": accuracy(per_q[~big], m)}
    if f"{m} | count" in per_q:
        row.update(count_error(per_q, m))
    table.append(row)
counting = pd.DataFrame(table)
print(counting.to_string(index=False))

def mcnemar(df, a="chain_s42", b="R11"):
    """Exact McNemar: only the questions one method gets right and the other wrong count."""
    x, y = df[f"{a} | correct"], df[f"{b} | correct"]
    only_a, only_b = int((x & ~y).sum()), int((~x & y).sum())
    p = binomtest(only_a, only_a + only_b, 0.5).pvalue if only_a + only_b else 1.0
    return {"questions": len(df), "only_chain_right": only_a, "only_R11_right": only_b, "p": round(float(p), 4)}

paired = {"all": mcnemar(per_q), "clean": mcnemar(per_q[per_q["clean"]])}
print("\nChain with grader seed 42 against R11, same questions (exact McNemar):", paired)


✓ The true masks answer all 1,055 questions; R11 reproduces 39.24%
       method  all_%  clean_%  intact_%  damaged_%  destroyed_%  largest_above_20_%  largest_up_to_20_%  mean_abs_error  median_abs_error  median_error_%  within_20%_%
    chain_s42  37.73    37.47     37.48      34.51        42.59               22.78               44.77            7.65               3.0            40.0          32.8
    chain_s43  37.63    36.55     35.32      40.49        39.81               22.78               44.63            7.62               3.0            37.5          32.5
    chain_s44  38.67    37.99     37.84      34.51        46.30               24.26               45.47            7.36               3.0            36.4          34.8
          R11  39.24    39.01     38.02      43.66        36.57               27.51               44.77            7.38               2.0            33.3          38.8
no_image_rule  59.72    60.06     65.41      54.23        52.31              100.00          

In [4]:
# ── Cell 4: Against the expectation, and the output files ───────────────────
def as_markdown(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:              # `tabulate` missing: a plain-text table instead
        return "```\n" + df.to_string(index=False) + "\n```"

acc = {s: counting.loc[counting["method"] == f"chain_s{s}", "all_%"].item() for s in BIASES}
in_range = 36 <= acc[42] <= 40
err42 = counting.loc[counting["method"] == "chain_s42", "median_error_%"].item()
err_r11 = counting.loc[counting["method"] == "R11", "median_error_%"].item()
finding = (f"The chain with grader seed 42 answers {acc[42]}% of the 1,055 validation questions, against {r11_acc}% for R11 "
           f"(exact McNemar p = {paired['all']['p']}). Median count error: {err42}% against {err_r11}%. "
           f"Expected about 36-40%: {'met' if in_range else 'NOT met'}.")
print(finding)
per_q.to_csv(OUT_DIR / "e7_step1_per_question.csv", index=False)
counting.to_csv(OUT_DIR / "e7_step1_counting.csv", index=False)
check.to_csv(OUT_DIR / "e7_step1_map_rebuild_check.csv", index=False)
config = {"task": "E7 step 1", "questions": len(per_q), "validation_images": len(val_files), "min_area": MIN_AREA,
          "biases": {str(s): b.tolist() for s, b in BIASES.items()}, "paired_with_r11": paired, "finding": finding,
          "run_time_min": round((time.time() - t_start) / 60, 1)}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=1), encoding="utf-8")
(OUT_DIR / "README.md").write_text("\n".join([
    "# E7 step 1: the chain's first counting number, on the validation images", "",
    "The chain: SAM 3 (prompt \"building\", on the before-photo in four quarters) finds each building; a Siamese ResNet-18 trained "
    "on SAM 3's outlines labels it intact, damaged, destroyed or not a building. R11 is the U-Net with a ResNet-50 encoder and "
    "before and after photos as input, the baseline. Each method counts 8-connected blobs of at least 20 px and picks the nearest "
    "option, on the 1,055 training-split counting questions on the 645 validation images. The true masks answer all of them.", "",
    f"**Finding:** {finding}", "",
    "## Accuracy and count error", "", as_markdown(counting), "",
    "- `clean_%`: without E5's 40 leaked validation pairs.",
    "- `largest_above_20_%` and `largest_up_to_20_%`: questions by their largest option. When it's above 20, the options are "
    "built around the correct count, so the middle value is correct.",
    "- `no_image_rule` and `always_middle` never see the image. They were found on these training questions, so here they are "
    "only a reference. The fair test of the rule is `benchmark-option-rule-check.ipynb`, on the benchmark's questions.",
    "- Count error is against the question's true count: `median_error_%` as in R16c step 1, `within_20%_%` the share of counts "
    "within 20% of the truth.", "",
    "## Chain with grader seed 42 against R11, same questions (exact McNemar)", "",
    f"- All: {paired['all']}", f"- Clean: {paired['clean']}", "",
    "## The rebuilt maps against their published scores", "", as_markdown(check), "",
    "## Files", "",
    "- `e7_step1_counting.csv`, `e7_step1_per_question.csv`, `e7_step1_map_rebuild_check.csv`, `config.json`",
]) + "\n", encoding="utf-8")
print(f"✓ Wrote README.md, the CSVs and config.json to {OUT_DIR} in {config['run_time_min']} min")


The chain with grader seed 42 answers 37.73% of the 1,055 validation questions, against 39.24% for R11 (exact McNemar p = 0.3881). Median count error: 40.0% against 33.3%. Expected about 36-40%: met.
✓ Wrote README.md, the CSVs and config.json to D:\CSE499AB_project\Essential_task_delegation\Champion_Task\E7-results\step1 in 2.7 min
